# G1 and G3 Generation Demo

## Goal and scope
Demonstrate direct generation (G1) and observed-human-analysis-mediated generation (G3) on the same approved R1 development documents. This notebook is the G1/G3 entry point; reusable generation, validation, and evaluation functions live in importable project modules. The separate G2 notebook consumes predicted analysis through the same request and output contracts.

## Conditions
- **G1: Direct baseline.** Provide ordered reflection segments and the rubric, with no segment analysis.
- **G2: Predicted-analysis intervention.** Provide the same reflection and rubric plus predicted scope and component-band labels.
- **G3: Observed human analysis.** Provide the same reflection and rubric plus the available candidate-grain human annotations. These may be incomplete, ambiguous, or marked for review; this development condition is not an oracle.

Use the same generator model, shared prompt content, output schema, and generation settings across conditions. Keep reflection text, annotations, model outputs, and traces within approved protected storage.

## 1. Select the R1 development cohort

The shared `config/experiment.toml` selects all R1 documents in the held-out test split. Candidate annotations are retained as recorded, including review flags, competing candidate bundles, and missing values. G3 uses these as imperfect human context; no adjudication or label imputation is done in this batch.

### Load the configured development documents

The selection cell joins R1 records from the human-feedback source with documents in the canonical test modeling split. It checks reflection text and candidate segment consistency but does not filter out review-marked or ambiguous human annotations.

In [6]:
from pathlib import Path
import importlib

import pandas as pd

from reflection_assessment_feedback import EXPERT_RUBRIC
import reflection_assessment_feedback.data as reflection_data
from reflection_assessment_feedback.experiment_config import load_experiment_config

reflection_data = importlib.reload(reflection_data)
load_human_analysis_context = reflection_data.load_human_analysis_context

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
EXPERIMENT_CONFIG = load_experiment_config()
DATASET_CONFIG = EXPERIMENT_CONFIG["dataset"]
DEVELOPMENT_ESSAY_NAME = DATASET_CONFIG["development_essay_name"]
DATA_PATH = (
    PROJECT_ROOT
    / EXPERIMENT_CONFIG["paths"]["provisional_split_directory"]
    / f"provisional_{DATASET_CONFIG['prediction_split']}_modeling_wide.csv"
)
FEEDBACK_PATH = (PROJECT_ROOT / EXPERIMENT_CONFIG["paths"]["human_feedback"]).resolve()
if not FEEDBACK_PATH.is_file():
    raise FileNotFoundError("The protected testset feedback copy was not found.")

feedback_rows = pd.read_csv(FEEDBACK_PATH, dtype={"document_id": str})
required_feedback_columns = {"essay_name", "document_id", "full_text", "feedback_text"}
if not required_feedback_columns.issubset(feedback_rows.columns):
    raise ValueError("The testset feedback copy is missing required columns.")
feedback_rows["essay_name"] = feedback_rows["essay_name"].astype(str).str.strip().str.upper()
feedback_rows["document_id"] = feedback_rows["document_id"].astype(str).str.strip()
if feedback_rows["document_id"].duplicated().any():
    raise ValueError("The testset feedback copy must have one row per document.")

split_document_ids = set(
    pd.read_csv(DATA_PATH, usecols=["document_id"], dtype={"document_id": str})[
        "document_id"
    ].dropna()
)
development_feedback = (
    feedback_rows.loc[
        (feedback_rows["essay_name"] == DEVELOPMENT_ESSAY_NAME)
        & feedback_rows["document_id"].isin(split_document_ids)
    ]
    .sort_values("document_id")
    .reset_index(drop=True)
)
if development_feedback.empty:
    raise ValueError("No development-cohort documents were found in the test split.")

DEVELOPMENT_BATCH = []
for feedback_record in development_feedback.to_dict(orient="records"):
    if not str(feedback_record["full_text"]).strip() or not str(feedback_record["feedback_text"]).strip():
        raise ValueError("A development document is missing its full text or human feedback.")
    document, human_context = load_human_analysis_context(
        DATA_PATH,
        feedback_record["document_id"],
    )
    DEVELOPMENT_BATCH.append(
        {
            "document": document,
            "human_context": human_context,
            "human_feedback_record": feedback_record,
        }
    )
print(
    f"Loaded {len(DEVELOPMENT_BATCH)} {DEVELOPMENT_ESSAY_NAME} development documents; "
    f"segments={sum(len(item['document'].segments) for item in DEVELOPMENT_BATCH)}, "
    f"candidate bundles={sum(sum(len(segment.candidates) for segment in item['human_context'].segments) for item in DEVELOPMENT_BATCH)}."
)


Loaded 6 R1 development documents; segments=282, candidate bundles=288.


### 2. Check the batch contents

The next cell reports aggregate document, segment, candidate-annotation, and review-flag counts only. It does not display reflection text, candidate labels, or identifiers.

In [7]:
candidate_bundle_count = sum(
    len(segment.candidates)
    for item in DEVELOPMENT_BATCH
    for segment in item["human_context"].segments
)
reviewed_segment_count = sum(
    any(candidate.requires_review for candidate in segment.candidates)
    for item in DEVELOPMENT_BATCH
    for segment in item["human_context"].segments
)
print(
    f"Development cohort: {len(DEVELOPMENT_BATCH)} documents, "
    f"{sum(len(item['document'].segments) for item in DEVELOPMENT_BATCH)} segments, "
    f"{candidate_bundle_count} candidate annotations, "
    f"{reviewed_segment_count} segments with review flags."
)

Development cohort: 6 documents, 282 segments, 288 candidate annotations, 6 segments with review flags.


### 3. Set tracing policy

The shared `config/experiment.toml` enables LangSmith tracing for this approved R1 development batch. Configure `LANGSMITH_API_KEY` and `LANGSMITH_PROJECT` in `.env`, and ensure the EU project’s access and retention policy are approved. Traces may contain full reflections, prompts, imperfect candidate analysis, and generated outputs. `APPROVE_EXTERNAL_PROCESSING` remains a separate runtime gate.

In [8]:
from reflection_assessment_feedback.experiment_config import load_experiment_config

EXPERIMENT_CONFIG = load_experiment_config()
ENABLE_LANGSMITH_TRACING = EXPERIMENT_CONFIG["generation"]["enable_langsmith_tracing"]

### 4. Generate paired G1/G3 development runs

After confirming provider approval, this cell runs one G1 and one G3 generation for each R1 document. G3 receives the observed human candidate bundles, review flags, and unknowns; it is not an oracle condition. The shared limiter spaces Gemini requests by at least 7.5 seconds, and tracing follows the approved shared setting.

In [9]:
import importlib
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

from reflection_assessment_feedback.experiment_config import (
    experiment_config_sha256,
    load_experiment_config,
)
from reflection_assessment_feedback.prompt_registry import resolve_prompt
from reflection_assessment_feedback.rate_limit import reserve_request_slot

EXPERIMENT_CONFIG = load_experiment_config()
GENERATION_CONFIG = EXPERIMENT_CONFIG["generation"]
GENERATION_PROMPT = resolve_prompt(
    GENERATION_CONFIG["prompt_id"],
    GENERATION_CONFIG["expected_prompt_version"],
)
langsmith_dotenv_path = PROJECT_ROOT / ".env"
langsmith_dotenv = dotenv_values(langsmith_dotenv_path)
load_dotenv(langsmith_dotenv_path, override=False)
conflicting_langsmith_settings = [
    name
    for name in ("LANGSMITH_API_KEY", "LANGSMITH_PROJECT", "LANGSMITH_ENDPOINT")
    if langsmith_dotenv.get(name)
    and os.getenv(name)
    and langsmith_dotenv[name] != os.getenv(name)
]
if conflicting_langsmith_settings:
    raise RuntimeError(
        "LangSmith settings differ between .env and the active kernel environment: "
        f"{', '.join(conflicting_langsmith_settings)}. The kernel environment takes "
        "precedence; update or unset the conflicting value and restart the kernel. "
        "Secret values are not displayed."
    )

from langchain_google_genai import ChatGoogleGenerativeAI
import reflection_assessment_feedback.generator as generation_module
from reflection_assessment_feedback import make_generation_request

generation_module = importlib.reload(generation_module)
RubricGenerationRunner = generation_module.RubricGenerationRunner
if generation_module.PROMPT_VERSION != GENERATION_PROMPT.version:
    raise ValueError("Runtime generator prompt differs from the selected prompt artifact.")
if EXPERT_RUBRIC.version != GENERATION_CONFIG["expected_rubric_version"]:
    raise ValueError("Rubric version differs from experiment config.")

APPROVE_EXTERNAL_PROCESSING = True
REPETITIONS = GENERATION_CONFIG["repetitions"]
assert APPROVE_EXTERNAL_PROCESSING, "Confirm provider processing is approved before running."
assert os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"), (
    "Configure GOOGLE_API_KEY or GEMINI_API_KEY in the approved environment."
)
if ENABLE_LANGSMITH_TRACING:
    assert os.getenv("LANGSMITH_API_KEY"), (
        "Configure LANGSMITH_API_KEY in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_PROJECT"), (
        "Configure LANGSMITH_PROJECT in the local .env or approved environment."
    )
    assert os.getenv("LANGSMITH_ENDPOINT", "https://eu.api.smith.langchain.com").rstrip("/") == (
        "https://eu.api.smith.langchain.com"
    ), "Tracing is restricted to the LangSmith EU endpoint."

protected_root_value = os.getenv("PREBI_DATA_ROOT")
if not protected_root_value:
    raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before running the batch.")
protected_root = Path(protected_root_value).expanduser().resolve()
if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")
rate_limit_state = protected_root / ".rate_limits" / "gemini.state"

chat_model = ChatGoogleGenerativeAI(
    model=GENERATION_CONFIG["model_name"],
    temperature=GENERATION_CONFIG["temperature"],
)
runner = RubricGenerationRunner(
    chat_model,
    enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
)
RUNS_BY_DOCUMENT = {}
for batch_item in DEVELOPMENT_BATCH:
    batch_document = batch_item["document"]
    g1_request = make_generation_request(
        condition="G1",
        document=batch_document,
        rubric=EXPERT_RUBRIC,
    )
    g3_request = make_generation_request(
        condition="G3",
        document=batch_document,
        rubric=EXPERT_RUBRIC,
        analysis=batch_item["human_context"],
    )
    condition_runs = {"G1": [], "G3": []}
    for condition, request in (("G1", g1_request), ("G3", g3_request)):
        for repetition in range(1, REPETITIONS + 1):
            reserve_request_slot(
                rate_limit_state,
                GENERATION_CONFIG["minimum_request_interval_seconds"],
            )
            condition_runs[condition].append(
                runner.generate(request, repetition=repetition)
            )
    RUNS_BY_DOCUMENT[batch_document.document_id] = condition_runs

print(
    f"Generated G1/G3 for {len(RUNS_BY_DOCUMENT)} R1 development documents; "
    f"Gemini request slots spaced by at least "
    f"{GENERATION_CONFIG['minimum_request_interval_seconds']} seconds."
)
if ENABLE_LANGSMITH_TRACING:
    print(
        "LangSmith tracing targeted EU endpoint "
        f"for project {os.environ['LANGSMITH_PROJECT']}; check warnings above for upload failures."
    )

/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/Users/quynguyen/Documents/CodingProjects/prebi_v1/.venv/lib/python3.14/site-packages/langchain_google_genai/c

Generated G1/G3 for 6 R1 development documents; Gemini request slots spaced by at least 7.5 seconds.
LangSmith tracing targeted EU endpoint for project prebi_v1; check warnings above for upload failures.


In [ ]:
{
    "documents": len(RUNS_BY_DOCUMENT),
    "paired_g1_g3_runs": sum(len(runs["G1"]) for runs in RUNS_BY_DOCUMENT.values()),
}

{'G1': [GenerationRun(run_id='2711dac9-e2f6-4e52-8cf3-514d1ef9c99d', condition='G1', repetition=1, model_name='gemini-3.5-flash-lite', generation_parameters={'temperature': 0.0}, prompt_version='1.4.0', output=GenerationOutput(assessment=Assessment(component_assessments=[DimensionAssessment(dimension_id='SW', band=2.2, justification='Die Verfasserin wählt eine konkrete Situation aus der Unterrichtsvignette aus – nämlich die Fragestellung zur Farbe in der Skizze und das anschließende Verweilen beziehungsweise frühe Eingreifen des Lehrenden – und beschreibt diese nachvollziehbar.', evidence_segment_ids=['segment-829bb01c3fe571c41b6b', 'segment-c16a465f722e05bbdd25', 'segment-2ed98c1528e7d233e732']), DimensionAssessment(dimension_id='UA', band=2.1, justification='Es wird eine grundsätzlich zielführende Analyse der Ursachen vorgenommen, indem die Wirkung von Wartezeiten und die Einschränkung durch vorgegebene Antwortmöglichkeiten beleuchtet werden. Die Einbettung in Fachliteratur und persö

### 5. Save paired runs per document

Each document gets a separate protected G1/G3 comparison artifact under `PREBI_DATA_ROOT/g1-g3/demo-runs/`. Artifacts retain the exact G3 candidate-grain human context, review/missingness counts, rubric and prompt provenance, and the experiment-config fingerprint. Writes are atomic and use restrictive file permissions.

In [10]:
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from reflection_assessment_feedback.experiment_config import experiment_config_sha256
from reflection_assessment_feedback.generator import output_for_review
from reflection_assessment_feedback.prompt_registry import resolve_prompt
from reflection_assessment_feedback.rubric import RUBRIC_ARTIFACT

PROJECT_ROOT = Path.cwd().resolve()
assert PROJECT_ROOT.name == "prebi_v1", "Run this notebook from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
protected_root_value = os.getenv("PREBI_DATA_ROOT")
if not protected_root_value:
    raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before saving demo runs.")
protected_root = Path(protected_root_value).expanduser()
if not protected_root.is_absolute():
    raise ValueError("PREBI_DATA_ROOT must be an absolute path.")
protected_root = protected_root.resolve()
if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
    raise ValueError("PREBI_DATA_ROOT must be outside the project source tree.")

run_directory = protected_root / "g1-g3" / "demo-runs"
run_directory.mkdir(parents=True, exist_ok=True, mode=0o700)
if not run_directory.resolve().is_relative_to(protected_root):
    raise ValueError("Demo-run storage must resolve within PREBI_DATA_ROOT.")
if set(RUNS_BY_DOCUMENT) != {item["document"].document_id for item in DEVELOPMENT_BATCH}:
    raise ValueError("Generated G1/G3 documents do not match the selected development cohort.")


def serialize_run(run, batch_document):
    prompt_artifact = resolve_prompt(GENERATION_CONFIG["prompt_id"], run.prompt_version)
    if prompt_artifact.sha256 != run.prompt_artifact_sha256:
        raise ValueError("Run prompt artifact hash does not match the registered prompt version.")
    if run.rubric_artifact_sha256 != RUBRIC_ARTIFACT.sha256:
        raise ValueError("Run rubric hash does not match the registered rubric version.")
    return {
        "run_id": run.run_id,
        "condition": run.condition,
        "repetition": run.repetition,
        "model_name": run.model_name,
        "generation_parameters": run.generation_parameters,
        "prompt_version": run.prompt_version,
        "prompt_artifact_sha256": run.prompt_artifact_sha256,
        "prompt_components": run.prompt_components,
        "rendered_prompt_sha256": run.rendered_prompt_sha256,
        "rubric_artifact_sha256": run.rubric_artifact_sha256,
        "output": output_for_review(run.output, batch_document),
        "feedback_letter": run.feedback_letter,
    }


SAVED_COMPARISONS = []
for batch_item in DEVELOPMENT_BATCH:
    batch_document = batch_item["document"]
    condition_runs = RUNS_BY_DOCUMENT[batch_document.document_id]
    if set(condition_runs) != {"G1", "G3"}:
        raise ValueError("Each development document must have paired G1 and G3 runs.")
    if any(len(condition_runs[name]) != REPETITIONS for name in ("G1", "G3")):
        raise ValueError("Each condition must contain exactly REPETITIONS generated runs.")
    runs_by_repetition = {
        condition: {run.repetition: run for run in runs}
        for condition, runs in condition_runs.items()
    }
    if set(runs_by_repetition["G1"]) != set(runs_by_repetition["G3"]):
        raise ValueError("G1 and G3 runs must have matching repetition numbers.")

    human_context = batch_item["human_context"]
    candidate_count = sum(len(segment.candidates) for segment in human_context.segments)
    reviewed_segment_count = sum(
        any(candidate.requires_review for candidate in segment.candidates)
        for segment in human_context.segments
    )
    multi_candidate_segment_count = sum(
        len(segment.candidates) > 1 for segment in human_context.segments
    )
    comparison_package = {
        "schema_version": "1.5",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "experiment_config_sha256": experiment_config_sha256(),
        "development_cohort": DEVELOPMENT_ESSAY_NAME,
        "document_id": batch_document.document_id,
        "segment_count": len(batch_document.segments),
        "rubric_id": EXPERT_RUBRIC.rubric_id,
        "rubric_version": EXPERT_RUBRIC.version,
        "rubric_artifact_sha256": RUBRIC_ARTIFACT.sha256,
        "g3_analysis_quality": {
            "kind": "observed_human_annotations",
            "candidate_annotation_count": candidate_count,
            "reviewed_segment_count": reviewed_segment_count,
            "multiple_candidate_segment_count": multi_candidate_segment_count,
        },
        "g3_human_analysis_context": human_context.model_dump(mode="json"),
        "pairs": [
            {
                "repetition": repetition,
                "G1": serialize_run(runs_by_repetition["G1"][repetition], batch_document),
                "G3": serialize_run(runs_by_repetition["G3"][repetition], batch_document),
            }
            for repetition in sorted(runs_by_repetition["G1"])
        ],
    }
    output_path = run_directory / (
        f"comparison-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex}.json"
    )
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=run_directory,
            prefix=".comparison-", suffix=".tmp", delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(comparison_package, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write("\n")
        os.chmod(temporary_path, 0o600)
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
    SAVED_COMPARISONS.append((output_path, comparison_package))

print(f"Saved paired G1/G3 artifacts for {len(SAVED_COMPARISONS)} R1 development documents.")

Saved paired G1/G3 artifacts for 6 R1 development documents.


In [11]:
if len(SAVED_COMPARISONS) != len(DEVELOPMENT_BATCH):
    raise ValueError("Not every R1 development document has a saved G1/G3 comparison.")
items_by_id = {item["document"].document_id: item for item in DEVELOPMENT_BATCH}
for comparison_path, saved_comparison in SAVED_COMPARISONS:
    batch_item = items_by_id[saved_comparison["document_id"]]
    batch_document = batch_item["document"]
    human_context = batch_item["human_context"]
    assert saved_comparison["schema_version"] == "1.5"
    assert saved_comparison["experiment_config_sha256"] == experiment_config_sha256()
    assert saved_comparison["rubric_artifact_sha256"] == RUBRIC_ARTIFACT.sha256
    assert saved_comparison["segment_count"] == len(batch_document.segments)
    assert saved_comparison["g3_human_analysis_context"] == human_context.model_dump(mode="json")
    assert len(saved_comparison["pairs"]) == REPETITIONS
    orders_by_id = {segment.segment_id: segment.order for segment in batch_document.segments}
    for pair in saved_comparison["pairs"]:
        assert pair["G1"]["condition"] == "G1"
        assert pair["G3"]["condition"] == "G3"
        assert pair["G1"]["repetition"] == pair["G3"]["repetition"] == pair["repetition"]
        for condition in ("G1", "G3"):
            run_data = pair[condition]
            assert len(run_data["prompt_artifact_sha256"]) == 64
            assert len(run_data["rendered_prompt_sha256"]) == 64
            assert len(run_data["rubric_artifact_sha256"]) == 64
            assert len(run_data["prompt_components"]) == (2 if condition == "G3" else 1)
            for assessment in run_data["output"]["assessment"]["component_assessments"]:
                assert assessment["evidence_segment_orders"] == [
                    orders_by_id[segment_id] for segment_id in assessment["evidence_segment_ids"]
                ]
    disk_copy = json.loads(comparison_path.read_text(encoding="utf-8"))
    assert disk_copy == saved_comparison
print("All R1 G1/G3 artifacts passed pairing, candidate-context, evidence, prompt-component, and config checks.")

All R1 G1/G3 artifacts passed pairing, coverage, candidate-context, evidence, and config checks.


### 6. Verify batch persistence

The final cell reports aggregate counts only. Full reflection text, human feedback, candidate annotations, generated outputs, and traces remain in approved protected storage.

In [12]:
print(
    {
        "development_documents": len(SAVED_COMPARISONS),
        "segments": sum(len(item["document"].segments) for item in DEVELOPMENT_BATCH),
        "candidate_annotations": sum(
            len(segment.candidates)
            for item in DEVELOPMENT_BATCH
            for segment in item["human_context"].segments
        ),
        "paired_g1_g3_runs": sum(
            len(package["pairs"]) for _, package in SAVED_COMPARISONS
        ),
        "artifacts_validated": len(SAVED_COMPARISONS),
    }
)

{'development_documents': 6, 'segments': 282, 'candidate_annotations': 288, 'paired_g1_g3_runs': 6, 'artifacts_validated': 6}
